### Links and Resources
| Function | Use |
|---|---|
| [`count`, `count_distinct`, `approx_count_distinct`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.approx_count_distinct.html) | Counting, exact and approximate |
| [`sum`, `avg`, `min`, `max`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.sum.html) | The basics |
| [`collect_list`, `collect_set`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.collect_list.html) | Aggregate to an array |
| [`percentile_approx`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.percentile_approx.html) | Quantiles (approximate) |
| [`stddev`, `variance`, `corr`, `covar_samp`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.stddev.html) | Statistics |
| [GROUPING SETS / CUBE / ROLLUP](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-groupby.html) | Multi-level grouping |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

from pyspark.sql import functions as F

# Aggregation Functions Deep Dive

`groupBy().agg(...)` is one of the most-used patterns in Spark. The basics (`sum`, `count`, `avg`) are obvious; what this notebook covers is everything else: distinct counts, percentiles, list aggregates, statistics, and the `CUBE`/`ROLLUP` family for multi-level summaries.

In [ ]:
sales = spark.createDataFrame([
    ("ACME", "EU", "2024-01-01", "alice",   100),
    ("ACME", "EU", "2024-01-02", "bob",     150),
    ("ACME", "EU", "2024-01-02", "alice",   120),
    ("ACME", "US", "2024-01-01", "carol",   200),
    ("ACME", "US", "2024-01-03", "bob",      80),
    ("BETA", "EU", "2024-01-01", "alice",   300),
    ("BETA", "EU", "2024-01-02", "dave",    250),
    ("BETA", "US", "2024-01-03", "carol",   400),
    ("BETA", "US", "2024-01-04", "carol",   350),
], ["store", "region", "date", "customer", "amount"])
display(sales)

## The basics — `sum`, `avg`, `min`, `max`, `count`

Two `count` flavors:

- `count("*")` / `count(F.lit(1))` — counts rows including those with NULLs.
- `count("col")` — counts only **non-null** values in that column.

In [ ]:
sales.groupBy("store").agg(
    F.count("*").alias("rows"),
    F.sum("amount").alias("total"),
    F.avg("amount").alias("avg"),
    F.min("amount").alias("min"),
    F.max("amount").alias("max"),
).show()

## Distinct counts — exact vs approximate

`count_distinct` is **expensive** at scale: it requires a shuffle keyed on the column to deduplicate. For analytics dashboards over billions of rows, **`approx_count_distinct`** uses HyperLogLog++ and is dramatically faster, with a tunable error rate (default ~5%).

Use exact only when correctness demands it (audit, billing). Use approximate for "how many unique users today".

In [ ]:
sales.groupBy("store").agg(
    F.count_distinct("customer").alias("unique_customers_exact"),
    F.approx_count_distinct("customer", rsd=0.05).alias("unique_customers_approx"),
).show()

## `collect_list` and `collect_set` — aggregate to an array

- `collect_list` — preserves duplicates and order is *not* guaranteed.
- `collect_set` — deduplicates, also unordered.

Both materialize the whole group in memory on one executor, so watch out for huge groups.

In [ ]:
sales.groupBy("store").agg(
    F.collect_list("customer").alias("customer_list"),
    F.collect_set("customer").alias("customer_set"),
).show(truncate=False)

### Combining `collect_list` with `sort_array` to get a deterministic order

If you need the list ordered, sort by another column *before* collecting (Spark 3.0+ keeps order from the input partition only when run with `spark.sql.execution.sortBeforeRepartition=true`, which is fragile). Safer: collect a struct and sort the array afterwards.

In [ ]:
sales.groupBy("store").agg(
    F.array_sort(
        F.collect_list(F.struct("date", "customer", "amount"))
    ).alias("events_sorted")
).show(truncate=False)

## Percentiles — `percentile_approx`

Median, quartiles, p99 latency. Pass a single percentile or an array.

In [ ]:
sales.groupBy("store").agg(
    F.percentile_approx("amount", 0.5).alias("median"),
    F.percentile_approx("amount", [0.25, 0.5, 0.75, 0.99]).alias("quartiles_p99"),
).show(truncate=False)

## Statistics — `stddev`, `variance`, `skewness`, `kurtosis`, `corr`, `covar_samp`

`stddev` / `variance` are the *sample* versions (Bessel's correction). Use `stddev_pop` / `var_pop` for population statistics.

In [ ]:
sales.groupBy("store").agg(
    F.stddev("amount").alias("stddev"),
    F.variance("amount").alias("variance"),
    F.skewness("amount").alias("skewness"),
).show()

# Pairwise stats — same group key, two columns
sales.groupBy("store").agg(
    F.corr(F.length("customer"), "amount").alias("namelen_amount_corr"),
).show()

## Conditional aggregates — `sum(when(...))`

The classic trick: count or sum only rows that match a condition, without splitting into multiple queries.

In [ ]:
sales.groupBy("store").agg(
    F.sum(F.when(F.col("region") == "EU", F.col("amount"))).alias("eu_total"),
    F.sum(F.when(F.col("region") == "US", F.col("amount"))).alias("us_total"),
    F.count(F.when(F.col("amount") > 200, 1)).alias("big_orders"),
).show()

## `agg` with a dict — quick & dirty

For one-off summaries with default column names, you can pass a dict of `column -> agg_name`:

In [ ]:
sales.groupBy("store").agg({
    "amount":   "sum",
    "customer": "count",
    "*":        "count",
}).show()

## Multi-level totals — `CUBE`, `ROLLUP`, `GROUPING SETS`

Three ways to compute aggregates at multiple levels in a single pass:

- **`ROLLUP(a, b, c)`** — `(a, b, c)`, `(a, b)`, `(a)`, `()` — hierarchical totals.
- **`CUBE(a, b, c)`** — every subset — full cross-tab.
- **`GROUPING SETS((a, b), (a), ())`** — explicit list of groupings.

Rows where a key is "the grand total" have `NULL` in that column. Use `grouping(col)` to tell apart "real NULL" from "this is a subtotal row".

In [ ]:
sales.rollup("store", "region").agg(
    F.sum("amount").alias("total"),
    F.grouping("store").alias("g_store"),
    F.grouping("region").alias("g_region"),
).orderBy("g_store", "g_region", "store", "region").show()

In [ ]:
sales.cube("store", "region").agg(F.sum("amount").alias("total")) \
     .orderBy("store", "region").show()

### `GROUPING SETS` in SQL — the most flexible form

In [ ]:
sales.createOrReplaceTempView("sales")

spark.sql("""
SELECT store, region, SUM(amount) AS total
FROM sales
GROUP BY GROUPING SETS (
    (store, region),
    (store),
    (region),
    ()
)
ORDER BY store, region
""").show()

## `pivot` — wide-format aggregates

A specialised aggregation that turns distinct values of a column into new columns. Always pass the explicit value list — without it, Spark runs an extra pass to discover them.

In [ ]:
sales.groupBy("store").pivot("region", ["EU", "US"]).agg(F.sum("amount")).show()

## Performance notes

- Distinct counts shuffle by the column being counted — slow with high-cardinality data. Reach for `approx_count_distinct` first.
- `collect_list` / `collect_set` move all values for one group to a single executor — bound the group size or it OOMs.
- Aggregations are partial-aggregated before shuffle (a "combine" step), so `sum`/`count`/`avg` over big data are cheap.
- `count(distinct ...)` cannot partial-aggregate — that's why it's the slow one. `approx_count_distinct` *can* partial-aggregate, hence the speedup.

## Takeaways

- `count("*")` counts rows; `count("col")` counts non-NULLs.
- `approx_count_distinct` is the right default for distinct counts at scale.
- `collect_list` / `collect_set` are powerful but bounded by group size.
- `percentile_approx` for medians/quartiles/p99 — the exact `percentile` function exists but is far slower.
- `sum(when(cond, x))` lets you compute many conditional metrics in one pass.
- `ROLLUP` / `CUBE` / `GROUPING SETS` give multi-level totals without UNION ALL gymnastics.